# Honeydew AI — облачное обучение

Запускайте ячейки по порядку. Датасет собирается из 10 000 оригинальных синтетических паттернов и до 10 000 MIDI-перформансов из официального E-GMD (CC BY 4.0). E-GMD содержит партии ударных, а не полные многодорожечные песни. Скрипт проверяет SHA-256 архива и сохраняет сведения об источнике и лицензии.


In [ ]:
!nvidia-smi || true
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())


In [ ]:
!git clone --depth 1 --branch ai-beatmaker https://github.com/djdhsvsbdb-design/honeydew.git
%cd honeydew
!python -m pip install -r tools/ai/requirements.txt


In [ ]:
# Скачать проверенный E-GMD MIDI и собрать смешанный датасет 10k + 10k
!python tools/ai/download_egmd.py --limit 10000
!python tools/ai/build_dataset.py --synthetic-songs 10000 --external-limit 10000 --external-manifest data/ai/external/egmd/external-manifest.jsonl --out data/ai/honeydew-20k.jsonl
!wc -l data/ai/honeydew-20k.jsonl


In [ ]:
!python tools/ai/train_music.py --data data/ai/honeydew-20k.jsonl --epochs 20 --batch-size 8 --block 256 --val-fraction 0.1 --out /content/honeydew-music.pt


In [ ]:
!python tools/ai/generate_music.py --checkpoint /content/honeydew-music.best.pt --style trap --bars 8 --bpm 145 --out /content/honeydew-generated


In [ ]:
from google.colab import files
from pathlib import Path

for filename in ["/content/honeydew-music.pt", "/content/honeydew-music.best.pt", "/content/honeydew-generated.mid", "/content/honeydew-generated.json"]:
    if Path(filename).exists():
        files.download(filename)
    else:
        print(f"Пока нет файла: {filename}")
